# Official RNAup Production Notebook (`pfScale = 1.00`)

This notebook generates the **official RNAup dataset** for the project under a single fixed configuration.

## Scope
- Accessibility mode only
- Fixed `pfScale = 1.00`
- Fixed thermodynamic settings:
  - `-u 1-20`
  - `-c SHIME`
  - `-T 37`
  - `--salt 1.021`
  - `-d2`
- Target datasets:
  - `CRW`
  - `bpRNA-new`

## Main goals
1. Re-run RNAup from scratch under the final fixed setting.
2. Save **raw**, **parsed long**, and **parsed wide** outputs in a clean official directory.
3. Run lightweight QC to confirm that:
   - no warnings were emitted,
   - no failures occurred,
   - no expected outputs are missing,
   - core features were populated.

## Notes
- This is a **production notebook**, not an exploratory notebook.
- Explanatory text and code comments are written in English for third-party readability.
- Existing exploratory runs are treated only as optional reference data for lightweight comparison.


In [1]:
from pathlib import Path
import os
import re
import json
import time
import shutil
import hashlib
import platform
import subprocess
from datetime import datetime, timezone
from typing import Optional

import numpy as np
import pandas as pd
from IPython.display import display

def find_project_root(start: Optional[Path] = None) -> Path:
    """Find the project root by searching for the expected top-level directories.

    The notebook is expected to live under <project>/notebooks/. We do not require the
    results directory to exist in advance because this official notebook can create it.
    """
    if start is None:
        start = Path.cwd()

    required_names = ["data", "notebooks"]
    optional_markers = ["env", "README.md"]

    for candidate in [start] + list(start.parents):
        if all((candidate / name).exists() for name in required_names):
            if any((candidate / marker).exists() for marker in optional_markers):
                return candidate
            return candidate

    raise FileNotFoundError(
        "Project root could not be detected. "
        "Please place this notebook under <project>/notebooks/ "
        "or run it from inside the project tree."
    )

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
RESULTS_DIR = PROJECT_ROOT / "results"
OFFICIAL_DIR = RESULTS_DIR / "rnaup_official_pf1p00"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
OFFICIAL_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT =", PROJECT_ROOT)
print("DATA_DIR     =", DATA_DIR)
print("RESULTS_DIR  =", RESULTS_DIR)
print("OFFICIAL_DIR =", OFFICIAL_DIR)


PROJECT_ROOT = /Users/hatanaka/rnaup-official-pf1p00
DATA_DIR     = /Users/hatanaka/rnaup-official-pf1p00/data
RESULTS_DIR  = /Users/hatanaka/rnaup-official-pf1p00/results
OFFICIAL_DIR = /Users/hatanaka/rnaup-official-pf1p00/results/rnaup_official_pf1p00


## Official run configuration

This section fixes all runtime parameters used in the final production run.
The goal is to make the official outputs fully reproducible and easy to audit.


In [2]:
# Official dataset definitions
DATASETS = [
    {
        "name": "CRW",
        "input_csv": DATA_DIR / "crw_from_bpseq_acgu.csv",
        "timeout_sec_per_seq": 3600,
    },
    {
        "name": "bpRNA-new",
        "input_csv": DATA_DIR / "bprna_new_test_acgu.csv",
        "timeout_sec_per_seq": 1800,
    },
]

# Official RNAup settings
PFSCALE = 1.00
TEMP_C = 37.0
SALT_M = 1.021
DANGLES = 2
ULENGTH_SPEC = "1-20"
CONTRIB_SPEC = "SHIME"

# Accessibility mode only
USE_noLP = False
USE_b = False
USE_interaction_pairwise = False
USE_interaction_first = False

# Feature-generation settings
PRIMARY_WINDOW_LEN = 6
AUX_RANK_Z_ULENGTHS = [4, 6]
ALL_ULENGTHS = list(range(1, 21))
ALL_CONTRIBS = list("SHIME")
LOCAL_Z_RADIUS = 20
TOP_FRAC = 0.10

# Run-control settings
OVERWRITE_EXISTING_SEQUENCE_DIR = False
RESUME_FROM_EXISTING_RAW = True

# Use RNAup from the current environment PATH.
RNAUP_BIN = shutil.which("RNAup")
if RNAUP_BIN is None:
    raise FileNotFoundError(
        "RNAup was not found in PATH. "
        "Please activate the intended conda environment before running this notebook."
    )

def make_official_cmd() -> list[str]:
    """Build the single official RNAup command used throughout this notebook."""
    cmd = [
        RNAUP_BIN,
        "-u", ULENGTH_SPEC,
        "-c", CONTRIB_SPEC,
        "-T", str(TEMP_C),
        "--salt", str(SALT_M),
        "-d2",
        "--pfScale", str(PFSCALE),
        "--log-file=RNAup.log",
        "--log-time",
    ]
    if USE_noLP:
        cmd.append("--noLP")
    if USE_b:
        cmd.append("-b")
    if USE_interaction_pairwise:
        cmd.append("--interaction_pairwise")
    if USE_interaction_first:
        cmd.append("--interaction_first")
    return cmd

OFFICIAL_CMD = make_official_cmd()

print("Official RNAup command:")
print(" ".join(OFFICIAL_CMD))


Official RNAup command:
/opt/anaconda3/envs/rna6nt/bin/RNAup -u 1-20 -c SHIME -T 37.0 --salt 1.021 -d2 --pfScale 1.0 --log-file=RNAup.log --log-time


## Helper functions

The functions below implement:
- input validation,
- raw-file management,
- RNAup execution,
- raw output parsing,
- long-format generation,
- transcript-level rank and local z-score calculation,
- wide-format generation,
- QC summary generation.

The comments are intentionally explicit so that a third party can understand what each step is doing.


In [3]:
def sanitize_id(text: str) -> str:
    """Create a filesystem-safe sequence identifier."""
    base = re.sub(r"[^A-Za-z0-9_.-]+", "_", str(text)).strip("._")
    if not base:
        base = "seq"
    short_hash = hashlib.sha1(str(text).encode("utf-8")).hexdigest()[:10]
    return f"{base}__{short_hash}"

def sha256_text(text: str) -> str:
    """Return the SHA-256 hash of a text string."""
    return hashlib.sha256(text.encode("utf-8")).hexdigest()

def sha256_file(path: Path) -> str:
    """Return the SHA-256 hash of a file."""
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def command_hash(cmd: list[str]) -> str:
    """Create a stable hash for an RNAup command line."""
    return hashlib.sha256(" ".join(cmd).encode("utf-8")).hexdigest()

def list_dir_recursive(seq_dir: Path) -> list[dict]:
    """Record all files created for one sequence run for later auditing."""
    items = []
    for p in sorted(seq_dir.rglob("*")):
        items.append({
            "relpath": str(p.relative_to(seq_dir)),
            "is_dir": p.is_dir(),
            "size": None if p.is_dir() else p.stat().st_size,
        })
    return items

def read_input_csv(path: Path) -> pd.DataFrame:
    """
    Read and validate an input dataset.

    Required columns:
    - id
    - sequence
    - structure
    """
    df = pd.read_csv(path)
    required = {"id", "sequence", "structure"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Missing required columns in {path}: {missing}")

    df = df.copy()
    df["id"] = df["id"].astype(str)
    df["sequence"] = df["sequence"].astype(str).str.upper().str.replace("T", "U", regex=False)
    df["structure"] = df["structure"].astype(str)

    bad = df["sequence"].str.len() != df["structure"].str.len()
    if bad.any():
        raise ValueError(
            f"Sequence / structure length mismatch detected in {path}: {int(bad.sum())} rows"
        )

    return df

def detect_rnaup_outfile(seq_dir: Path) -> tuple[Optional[Path], Optional[str]]:
    """
    Detect the output file generated by RNAup.

    RNAup typically produces '*_up.out'. If that exact pattern is missing,
    this function falls back to '*.out' and records the reason.
    """
    preferred = sorted([p for p in seq_dir.rglob("*_up.out") if p.is_file()])
    if len(preferred) == 1:
        return preferred[0], None
    if len(preferred) > 1:
        return None, f"Multiple '*_up.out' files were found: {[str(p) for p in preferred]}"

    all_out = sorted([p for p in seq_dir.rglob("*.out") if p.is_file()])
    if len(all_out) == 1:
        return all_out[0], None
    if len(all_out) > 1:
        return None, f"Multiple '*.out' files were found: {[str(p) for p in all_out]}"

    return None, "No RNAup output file was found"

def parse_rnaup_out(out_path: Path) -> pd.DataFrame:
    """
    Parse a raw RNAup output file into a DataFrame.

    The parser expects a table header line starting with '# pos'.
    All numeric columns are converted to numeric dtype when possible.
    """
    lines = out_path.read_text(encoding="utf-8", errors="replace").splitlines()

    header_idx = None
    header_cols = None
    for i, line in enumerate(lines):
        if line.startswith("#"):
            stripped = line.lstrip("#").strip()
            if re.match(r"^pos(\s+|$)", stripped):
                header_idx = i
                header_cols = re.split(r"\s+", stripped)
                break

    if header_idx is None or header_cols is None:
        raise ValueError(f"RNAup table header could not be detected: {out_path}")

    data_rows = []
    for line in lines[header_idx + 1:]:
        if not line.strip():
            continue
        if line.startswith("#"):
            continue
        parts = re.split(r"\s+", line.strip())
        if len(parts) != len(header_cols):
            raise ValueError(
                f"Column count mismatch in {out_path}\n"
                f"header={header_cols}\n"
                f"row={parts}"
            )
        data_rows.append(parts)

    df = pd.DataFrame(data_rows, columns=header_cols)
    df["pos"] = pd.to_numeric(df["pos"], errors="coerce").astype("Int64")

    for col in df.columns:
        if col == "pos":
            continue
        df[col] = pd.to_numeric(df[col].replace("NA", np.nan), errors="coerce")

    return df

def melt_to_long(raw_df: pd.DataFrame, source_dataset: str, seq_id: str, seq_len: int, cmd_hash: str, rnaup_version: str) -> pd.DataFrame:
    """
    Convert the wide raw RNAup table into a normalized long table.

    One row represents:
    - one transcript region,
    - one unstructured length,
    - one contribution type (S/H/I/M/E).
    """
    value_cols = [c for c in raw_df.columns if re.fullmatch(r"u\d+[SHIME]", c)]
    if len(value_cols) == 0:
        raise ValueError("No u{length}{contribution} columns were found in the raw RNAup table")

    long_df = raw_df.melt(
        id_vars=["pos"],
        value_vars=value_cols,
        var_name="feature_key",
        value_name="open_energy",
    )
    long_df["ulength"] = long_df["feature_key"].str.extract(r"u(\d+)").astype(int)
    long_df["contribution"] = long_df["feature_key"].str.extract(r"u\d+([SHIME])")
    long_df = long_df.drop(columns=["feature_key"])
    long_df = long_df.rename(columns={"pos": "region_right_end_1based"})

    long_df.insert(0, "source_dataset", source_dataset)
    long_df.insert(1, "group_id", seq_id)
    long_df.insert(2, "id", seq_id)
    long_df.insert(3, "sequence_length", seq_len)
    long_df["run_command_hash"] = cmd_hash
    long_df["rnaup_version"] = rnaup_version

    return long_df

def percentile_low_is_good(s: pd.Series) -> pd.Series:
    """
    Convert opening-energy values into a within-transcript percentile rank.

    Lower opening energy means easier opening, so the best region gets rank 1.0
    and the worst region gets rank 0.0.
    """
    out = pd.Series(np.nan, index=s.index, dtype=float)
    valid = s.notna()
    n = int(valid.sum())

    if n == 0:
        return out
    if n == 1:
        out.loc[valid] = 1.0
        return out

    ranks = s.loc[valid].rank(method="average", ascending=True)
    out.loc[valid] = 1.0 - (ranks - 1.0) / (n - 1.0)
    return out

def local_zscore(s: pd.Series, radius: int = 20) -> pd.Series:
    """
    Compute a local z-score using a symmetric window around each position.

    This is a purely transcript-internal derived feature and does not alter the raw data.
    """
    arr = pd.to_numeric(s, errors="coerce").to_numpy(dtype=float)
    out = np.full(len(arr), np.nan, dtype=float)

    for i, x in enumerate(arr):
        if np.isnan(x):
            continue
        lo = max(0, i - radius)
        hi = min(len(arr), i + radius + 1)
        neighborhood = arr[lo:hi]
        neighborhood = neighborhood[~np.isnan(neighborhood)]

        if len(neighborhood) < 2:
            out[i] = 0.0
        else:
            mu = neighborhood.mean()
            sd = neighborhood.std(ddof=0)
            out[i] = 0.0 if sd == 0 else (x - mu) / sd

    return pd.Series(out, index=s.index)

def add_rank_and_local_z(long_df: pd.DataFrame) -> pd.DataFrame:
    """
    Add transcript-internal percentile rank and local z-score to the long table.
    """
    long_df = long_df.sort_values(
        ["id", "ulength", "contribution", "region_right_end_1based"]
    ).reset_index(drop=True)

    long_df["rank_in_transcript"] = (
        long_df.groupby(["id", "ulength", "contribution"], group_keys=False)["open_energy"]
        .apply(percentile_low_is_good)
    )

    long_df["local_z"] = (
        long_df.groupby(["id", "ulength", "contribution"], group_keys=False)["open_energy"]
        .apply(lambda s: local_zscore(s, radius=LOCAL_Z_RADIUS))
    )

    return long_df


In [4]:
def build_window_table(df_seq: pd.DataFrame, source_dataset: str, window_len: int = 6) -> pd.DataFrame:
    """
    Build the main window-level table used for the wide output.

    The official wide output uses a 6-nt primary window table and joins RNAup-derived
    features using the right-end position of each unstructured region.
    """
    rows = []
    for _, row in df_seq.iterrows():
        seq_id = str(row["id"])
        seq = row["sequence"]
        struct = row["structure"]
        n = len(seq)

        if n < window_len:
            continue

        for start0 in range(0, n - window_len + 1):
            end0 = start0 + window_len
            window_struct = struct[start0:end0]
            ss_count = window_struct.count(".")

            rows.append({
                "source_dataset": source_dataset,
                "group_id": seq_id,
                "id": seq_id,
                "sequence_length": n,
                "window_start0": start0,
                "window_start1": start0 + 1,
                "window_end1": end0,
                "window_len": window_len,
                "window_seq": seq[start0:end0],
                "target_ss_count": ss_count,
                "target_all_ss_6of6": int(ss_count == 6),
                "target_ge5_ss": int(ss_count >= 5),
            })

    return pd.DataFrame(rows)

def make_wide_feature_tables(long_df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Create three pivot tables:
    - core opening-energy features for all u1-u20 and SHIME,
    - transcript-internal rank features for u4S and u6S,
    - transcript-internal local z-score features for u4S and u6S.
    """
    open_df = long_df.copy()
    open_df["feature_name"] = "rnaup_open" + open_df["ulength"].astype(str) + "_" + open_df["contribution"]

    open_wide = (
        open_df[["id", "region_right_end_1based", "feature_name", "open_energy"]]
        .pivot(index=["id", "region_right_end_1based"], columns="feature_name", values="open_energy")
        .reset_index()
        .rename(columns={"region_right_end_1based": "window_end1"})
    )

    rank_df = long_df[
        (long_df["contribution"] == "S") &
        (long_df["ulength"].isin(AUX_RANK_Z_ULENGTHS))
    ].copy()
    rank_df["feature_name"] = "rnaup_open" + rank_df["ulength"].astype(str) + "_rank_in_transcript"
    rank_wide = (
        rank_df[["id", "region_right_end_1based", "feature_name", "rank_in_transcript"]]
        .pivot(index=["id", "region_right_end_1based"], columns="feature_name", values="rank_in_transcript")
        .reset_index()
        .rename(columns={"region_right_end_1based": "window_end1"})
    )

    z_df = long_df[
        (long_df["contribution"] == "S") &
        (long_df["ulength"].isin(AUX_RANK_Z_ULENGTHS))
    ].copy()
    z_df["feature_name"] = "rnaup_open" + z_df["ulength"].astype(str) + "_local_z"
    z_wide = (
        z_df[["id", "region_right_end_1based", "feature_name", "local_z"]]
        .pivot(index=["id", "region_right_end_1based"], columns="feature_name", values="local_z")
        .reset_index()
        .rename(columns={"region_right_end_1based": "window_end1"})
    )

    return open_wide, rank_wide, z_wide

def make_best_region_flags(long_df: pd.DataFrame) -> pd.DataFrame:
    """
    Create optional best-region flags for u4S and u6S.

    These flags are lightweight derived annotations that may be useful later for inspection.
    """
    frames = []
    for u in AUX_RANK_Z_ULENGTHS:
        tmp = long_df[
            (long_df["ulength"] == u) &
            (long_df["contribution"] == "S")
        ][["id", "region_right_end_1based", "open_energy"]].copy()

        tmp["min_open"] = tmp.groupby("id")["open_energy"].transform("min")
        tmp[f"rnaup_best_region_flag_u{u}_S"] = (tmp["open_energy"] == tmp["min_open"]).astype(int)
        tmp = tmp[["id", "region_right_end_1based", f"rnaup_best_region_flag_u{u}_S"]]
        tmp = tmp.rename(columns={"region_right_end_1based": "window_end1"})
        frames.append(tmp)

    if len(frames) == 0:
        return pd.DataFrame(columns=["id", "window_end1"])

    out = frames[0]
    for tmp in frames[1:]:
        out = out.merge(tmp, how="outer", on=["id", "window_end1"])
    return out

def summarize_lengths(df: pd.DataFrame) -> dict:
    """Return a simple sequence-length summary for notebook display and metadata."""
    lengths = df["sequence"].str.len()
    return {
        "n_transcripts": int(len(df)),
        "min_length": int(lengths.min()) if len(lengths) else None,
        "median_length": float(lengths.median()) if len(lengths) else None,
        "max_length": int(lengths.max()) if len(lengths) else None,
    }

def make_dataset_dirs(root: Path, dataset_name: str) -> dict:
    """
    Construct all dataset-specific directories under the official output root.
    """
    dataset_root = root / dataset_name
    dirs = {
        "dataset_root": dataset_root,
        "raw_root": dataset_root / "rnaup_raw" / dataset_name,
        "long_root": dataset_root / "rnaup_parsed_long" / dataset_name,
        "wide_root": dataset_root / "rnaup_parsed_wide" / dataset_name,
        "meta_root": dataset_root / "meta",
        "snapshot_root": dataset_root / "input_snapshot",
    }
    for d in dirs.values():
        d.mkdir(parents=True, exist_ok=True)
    return dirs

def run_one_sequence(source_dataset: str, seq_id: str, sequence: str, rnaup_version: str, cmd: list[str], raw_root: Path, timeout_sec: int) -> dict:
    """
    Run one isolated RNAup job and save all raw outputs for later auditing.

    One sequence gets its own directory to avoid accidental output-file appending.
    """
    safe_id = sanitize_id(seq_id)
    seq_dir = raw_root / safe_id

    if seq_dir.exists() and OVERWRITE_EXISTING_SEQUENCE_DIR:
        shutil.rmtree(seq_dir)

    seq_dir.mkdir(parents=True, exist_ok=True)

    stdin_path = seq_dir / "stdin.fa"
    command_path = seq_dir / "command.txt"
    stdout_path = seq_dir / "stdout.txt"
    stderr_path = seq_dir / "stderr.txt"
    log_path = seq_dir / "RNAup.log"
    manifest_path = seq_dir / "manifest.json"
    parsed_raw_table_path = seq_dir / "parsed_raw_table.csv.gz"
    file_listing_path = seq_dir / "file_listing.json"

    fasta = f">{safe_id}\n{sequence}\n@\n"
    stdin_path.write_text(fasta, encoding="utf-8")
    command_path.write_text(" ".join(cmd) + "\n", encoding="utf-8")

    cmd_hash = command_hash(cmd)

    # Resume mode is kept because the official run may be interrupted and resumed later.
    if RESUME_FROM_EXISTING_RAW and manifest_path.exists():
        try:
            old_manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
            old_out = old_manifest.get("output_file_path")
            if old_manifest.get("exit_status") == 0 and old_out and Path(old_out).exists():
                raw_df = parse_rnaup_out(Path(old_out))
                raw_df.to_csv(parsed_raw_table_path, index=False, compression="gzip")
                long_df = melt_to_long(
                    raw_df,
                    source_dataset=source_dataset,
                    seq_id=seq_id,
                    seq_len=len(sequence),
                    cmd_hash=cmd_hash,
                    rnaup_version=rnaup_version,
                )
                return {
                    "manifest": old_manifest,
                    "raw_df": raw_df,
                    "long_df": long_df,
                    "resumed": True,
                }
        except Exception:
            pass

    # Use shell redirection for a CLI-like execution model.
    bash_cmd = "set -o pipefail; " + " ".join([f'"{x}"' if " " in str(x) else str(x) for x in cmd]) + " < stdin.fa > stdout.txt 2> stderr.txt"

    t0 = time.time()
    proc = subprocess.run(
        ["bash", "-lc", bash_cmd],
        cwd=seq_dir,
        text=True,
        timeout=timeout_sec,
    )
    runtime_sec = time.time() - t0

    stderr_text = stderr_path.read_text(encoding="utf-8", errors="replace") if stderr_path.exists() else ""
    log_text = log_path.read_text(encoding="utf-8", errors="replace") if log_path.exists() else ""

    warning_present = False
    warning_notes = []

    if re.search(r"warning|overflow|underflow|pfScale|pf_scale", stderr_text, flags=re.IGNORECASE):
        warning_present = True
        warning_notes.append("stderr contains warning/overflow/underflow/pfScale related text")

    if re.search(r"warning|overflow|underflow|pfScale|pf_scale", log_text, flags=re.IGNORECASE):
        warning_present = True
        warning_notes.append("RNAup.log contains warning/overflow/underflow/pfScale related text")

    out_path, out_detect_note = detect_rnaup_outfile(seq_dir)
    file_listing = list_dir_recursive(seq_dir)
    file_listing_path.write_text(json.dumps(file_listing, ensure_ascii=False, indent=2), encoding="utf-8")

    manifest = {
        "dataset": source_dataset,
        "sequence_id": seq_id,
        "group_id": seq_id,
        "sequence_length": len(sequence),
        "command": " ".join(cmd),
        "run_command_hash": cmd_hash,
        "RNAup_version": rnaup_version,
        "date_time_utc": datetime.now(timezone.utc).isoformat(),
        "exit_status": int(proc.returncode),
        "warning_present": bool(warning_present),
        "warning_notes": warning_notes,
        "output_file_path": None if out_path is None else str(out_path),
        "output_detection_note": out_detect_note,
        "stdout_path": str(stdout_path),
        "stderr_path": str(stderr_path),
        "log_path": str(log_path) if log_path.exists() else None,
        "stdin_fa_path": str(stdin_path),
        "command_txt_path": str(command_path),
        "parsed_raw_table_path": str(parsed_raw_table_path),
        "file_listing_path": str(file_listing_path),
        "runtime_sec": runtime_sec,
        "sequence_sha256": sha256_text(sequence),
        "pfScale_used": PFSCALE,
    }

    raw_df = pd.DataFrame()
    long_df = pd.DataFrame()

    # If the RNAup job ran but did not create a detectable output file, keep the run as failure-like metadata.
    if proc.returncode == 0 and out_path is not None and out_path.exists():
        manifest["output_file_sha256"] = sha256_file(out_path)
        raw_df = parse_rnaup_out(out_path)
        raw_df.to_csv(parsed_raw_table_path, index=False, compression="gzip")

        long_df = melt_to_long(
            raw_df,
            source_dataset=source_dataset,
            seq_id=seq_id,
            seq_len=len(sequence),
            cmd_hash=cmd_hash,
            rnaup_version=rnaup_version,
        )
    else:
        manifest["warning_present"] = True
        manifest["warning_notes"].append(
            f"Run did not produce a clean detectable output (returncode={proc.returncode}, output_file_path={manifest['output_file_path']})"
        )

    manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")

    return {
        "manifest": manifest,
        "raw_df": raw_df,
        "long_df": long_df,
        "resumed": False,
    }

def process_dataset(dataset_cfg: dict, official_root: Path, rnaup_version: str, cmd: list[str]) -> dict:
    """
    Run the full official workflow for one dataset:
    - raw RNAup runs,
    - manifest aggregation,
    - parsed long export,
    - parsed wide export,
    - dataset status summary.
    """
    dataset_name = dataset_cfg["name"]
    input_csv = dataset_cfg["input_csv"]
    timeout_sec = dataset_cfg["timeout_sec_per_seq"]

    dirs = make_dataset_dirs(official_root, dataset_name)
    df = read_input_csv(input_csv)

    # Save an input snapshot and transcript metadata for auditing.
    snapshot_path = dirs["snapshot_root"] / input_csv.name
    if not snapshot_path.exists():
        shutil.copy2(input_csv, snapshot_path)

    input_meta = df.copy()
    input_meta["group_id"] = input_meta["id"]
    input_meta["sequence_length"] = input_meta["sequence"].str.len()
    input_meta["sequence_sha256"] = input_meta["sequence"].map(sha256_text)
    input_meta.to_csv(dirs["meta_root"] / "input_transcript_metadata.csv", index=False)

    manifest_records = []
    long_frames = []

    for i, (_, row) in enumerate(df.iterrows(), start=1):
        # Print progress occasionally without depending on non-standard packages.
        if i == 1 or i % 10 == 0 or i == len(df):
            print(f"[{dataset_name}] {i}/{len(df)} : {row['id']}")

        result = run_one_sequence(
            source_dataset=dataset_name,
            seq_id=row["id"],
            sequence=row["sequence"],
            rnaup_version=rnaup_version,
            cmd=cmd,
            raw_root=dirs["raw_root"],
            timeout_sec=timeout_sec,
        )
        manifest_records.append(result["manifest"])
        if not result["long_df"].empty:
            long_frames.append(result["long_df"])

    manifest_df = pd.DataFrame(manifest_records)
    manifest_path = dirs["meta_root"] / "rnaup_run_manifest.csv"
    manifest_df.to_csv(manifest_path, index=False)

    if len(long_frames) == 0:
        raise RuntimeError(f"{dataset_name}: no long-format RNAup data were produced")

    long_df = pd.concat(long_frames, ignore_index=True)
    long_df = add_rank_and_local_z(long_df)

    long_path = dirs["long_root"] / "rnaup_long.csv.gz"
    long_df.to_csv(long_path, index=False, compression="gzip")

    base_wide_df = build_window_table(df, source_dataset=dataset_name, window_len=PRIMARY_WINDOW_LEN)
    open_wide, rank_wide, z_wide = make_wide_feature_tables(long_df)
    best_flag_wide = make_best_region_flags(long_df)

    wide_df = base_wide_df.merge(open_wide, how="left", on=["id", "window_end1"])
    wide_df = wide_df.merge(rank_wide, how="left", on=["id", "window_end1"])
    wide_df = wide_df.merge(z_wide, how="left", on=["id", "window_end1"])
    if not best_flag_wide.empty:
        wide_df = wide_df.merge(best_flag_wide, how="left", on=["id", "window_end1"])

    wide_path = dirs["wide_root"] / "rnaup_window_features.csv.gz"
    wide_df.to_csv(wide_path, index=False, compression="gzip")

    status = {
        "dataset_name": dataset_name,
        "input_csv": str(input_csv),
        "official_command": " ".join(cmd),
        "RNAup_version": rnaup_version,
        "pfScale_used": PFSCALE,
        "n_sequences_input": int(len(df)),
        "n_manifest_rows": int(len(manifest_df)),
        "n_success": int(((manifest_df["exit_status"] == 0) & manifest_df["output_file_path"].notna()).sum()),
        "n_failure": int((manifest_df["exit_status"] != 0).sum()),
        "n_warning": int(manifest_df["warning_present"].fillna(False).sum()),
        "n_missing_output": int(manifest_df["output_file_path"].isna().sum()),
        "n_long_rows": int(len(long_df)),
        "n_wide_rows": int(len(wide_df)),
        "raw_root": str(dirs["raw_root"]),
        "manifest_path": str(manifest_path),
        "long_path": str(long_path),
        "wide_path": str(wide_path),
    }
    dataset_status_path = dirs["meta_root"] / "dataset_status.json"
    dataset_status_path.write_text(json.dumps(status, ensure_ascii=False, indent=2), encoding="utf-8")

    return {
        "dataset_name": dataset_name,
        "dirs": dirs,
        "input_df": df,
        "manifest_df": manifest_df,
        "long_df": long_df,
        "wide_df": wide_df,
        "status": status,
    }

def make_dataset_qc(result: dict) -> pd.DataFrame:
    """
    Create a compact dataset-level QC table suitable for notebook display and file export.
    """
    dataset_name = result["dataset_name"]
    input_df = result["input_df"]
    manifest_df = result["manifest_df"]
    long_df = result["long_df"]
    wide_df = result["wide_df"]
    status = result["status"]

    core_cols = [
        "rnaup_open4_S",
        "rnaup_open6_S",
        "rnaup_open4_rank_in_transcript",
        "rnaup_open6_rank_in_transcript",
        "rnaup_open4_local_z",
        "rnaup_open6_local_z",
    ]

    rows = [
        {"dataset": dataset_name, "metric": "input_transcripts", "value": int(len(input_df))},
        {"dataset": dataset_name, "metric": "manifest_rows", "value": int(len(manifest_df))},
        {"dataset": dataset_name, "metric": "long_rows", "value": int(len(long_df))},
        {"dataset": dataset_name, "metric": "wide_rows", "value": int(len(wide_df))},
        {"dataset": dataset_name, "metric": "warnings", "value": int(status["n_warning"])},
        {"dataset": dataset_name, "metric": "failures", "value": int(status["n_failure"])},
        {"dataset": dataset_name, "metric": "missing_output", "value": int(status["n_missing_output"])},
    ]

    for col in core_cols:
        miss = float(wide_df[col].isna().mean()) if col in wide_df.columns and len(wide_df) > 0 else np.nan
        rows.append({"dataset": dataset_name, "metric": f"missing_rate::{col}", "value": miss})

    rows.append({
        "dataset": dataset_name,
        "metric": "input_vs_manifest_consistent",
        "value": int(len(input_df) == len(manifest_df)),
    })

    rows.append({
        "dataset": dataset_name,
        "metric": "clean_run",
        "value": int(status["n_warning"] == 0 and status["n_failure"] == 0 and status["n_missing_output"] == 0),
    })

    return pd.DataFrame(rows)

def safe_spearman_rank(x: pd.Series, y: pd.Series) -> float:
    """
    Compute a Spearman-like correlation using rank-transformed vectors.

    This avoids introducing extra dependencies and is sufficient for lightweight QC.
    """
    x = pd.to_numeric(x, errors="coerce")
    y = pd.to_numeric(y, errors="coerce")
    valid = x.notna() & y.notna()
    if valid.sum() < 2:
        return np.nan
    xr = x[valid].rank(method="average")
    yr = y[valid].rank(method="average")
    coef = xr.corr(yr, method="pearson")
    return float(coef) if pd.notna(coef) else np.nan

def topk_overlap(rank_a: pd.Series, rank_b: pd.Series, frac: float = 0.10) -> float:
    """
    Compute the overlap of the top-k regions defined by transcript-internal rank.
    """
    valid = rank_a.notna() & rank_b.notna()
    if valid.sum() == 0:
        return np.nan

    a = rank_a[valid]
    b = rank_b[valid]
    k = max(1, int(np.ceil(len(a) * frac)))

    top_a = set(a.sort_values(ascending=False).head(k).index.tolist())
    top_b = set(b.sort_values(ascending=False).head(k).index.tolist())

    return len(top_a & top_b) / k

def summarize_reference_comparison(new_long_df: pd.DataFrame, ref_long_path: Path, dataset_name: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Optionally compare the official pfScale=1.00 run to an existing reference long file.

    This is a lightweight sanity check only. The official outputs do not depend on the reference.
    """
    ref_long_df = pd.read_csv(ref_long_path)

    rows = []
    for seq_id in sorted(new_long_df["id"].unique()):
        rec = {"dataset_name": dataset_name, "id": seq_id}

        for u in [4, 6]:
            ref_u = ref_long_df[
                (ref_long_df["id"] == seq_id) &
                (ref_long_df["ulength"] == u) &
                (ref_long_df["contribution"] == "S")
            ][["region_right_end_1based", "open_energy", "rank_in_transcript"]].rename(
                columns={"open_energy": "open_ref", "rank_in_transcript": "rank_ref"}
            )

            new_u = new_long_df[
                (new_long_df["id"] == seq_id) &
                (new_long_df["ulength"] == u) &
                (new_long_df["contribution"] == "S")
            ][["region_right_end_1based", "open_energy", "rank_in_transcript"]].rename(
                columns={"open_energy": "open_new", "rank_in_transcript": "rank_new"}
            )

            merged = ref_u.merge(new_u, on="region_right_end_1based", how="inner").sort_values("region_right_end_1based")

            rec[f"n_common_u{u}"] = int(len(merged))
            rec[f"spearman_open{u}_S"] = safe_spearman_rank(merged["open_ref"], merged["open_new"])
            rec[f"spearman_rank_open{u}_S"] = safe_spearman_rank(merged["rank_ref"], merged["rank_new"])
            rec[f"top10pct_overlap_open{u}_S"] = topk_overlap(merged["rank_ref"], merged["rank_new"], frac=TOP_FRAC)

        rows.append(rec)

    detail_df = pd.DataFrame(rows)

    metric_cols = [
        "spearman_open4_S",
        "spearman_open6_S",
        "spearman_rank_open4_S",
        "spearman_rank_open6_S",
        "top10pct_overlap_open4_S",
        "top10pct_overlap_open6_S",
    ]
    summary_rows = []
    for col in metric_cols:
        s = pd.to_numeric(detail_df[col], errors="coerce")
        summary_rows.append({
            "dataset_name": dataset_name,
            "metric": col,
            "n": int(s.notna().sum()),
            "min": float(s.min()) if s.notna().any() else np.nan,
            "median": float(s.median()) if s.notna().any() else np.nan,
            "mean": float(s.mean()) if s.notna().any() else np.nan,
            "p05": float(s.quantile(0.05)) if s.notna().any() else np.nan,
            "p95": float(s.quantile(0.95)) if s.notna().any() else np.nan,
        })

    summary_df = pd.DataFrame(summary_rows)
    return detail_df, summary_df


## Input dataset validation

This section checks that both official input tables:
- exist,
- contain the required columns,
- have internally consistent sequence / structure lengths.

It also prints a compact transcript-length summary before any RNAup jobs are launched.


In [5]:
input_summaries = []
input_dfs = {}

for ds in DATASETS:
    df = read_input_csv(ds["input_csv"])
    input_dfs[ds["name"]] = df

    summary = summarize_lengths(df)
    summary["dataset_name"] = ds["name"]
    summary["input_csv"] = str(ds["input_csv"])
    input_summaries.append(summary)

input_summary_df = pd.DataFrame(input_summaries)[[
    "dataset_name", "input_csv", "n_transcripts", "min_length", "median_length", "max_length"
]]

display(input_summary_df)


,dataset_name,input_csv,n_transcripts,min_length,median_length,max_length
0,CRW,/Users/hatanaka/rnaup-official-pf1p00/data/crw...,113,953,2908.0,5184
1,bpRNA-new,/Users/hatanaka/rnaup-official-pf1p00/data/bpr...,5388,33,92.0,489


## Environment capture and official configuration export

This section records:
- RNAup version,
- Python version,
- platform,
- the official command line,
- the dataset list,
- the final output directory.

These files are meant to support long-term reproducibility.


In [6]:
version_proc = subprocess.run([RNAUP_BIN, "--version"], text=True, capture_output=True, check=True)
RNAUP_VERSION = version_proc.stdout.strip()

environment_info = {
    "python_version": platform.python_version(),
    "platform": platform.platform(),
    "RNAUP_BIN": RNAUP_BIN,
    "RNAup_version": RNAUP_VERSION,
    "project_root": str(PROJECT_ROOT),
    "data_dir": str(DATA_DIR),
    "results_dir": str(RESULTS_DIR),
    "official_dir": str(OFFICIAL_DIR),
}

run_config = {
    "official_output_root": str(OFFICIAL_DIR),
    "official_command": " ".join(OFFICIAL_CMD),
    "pfScale": PFSCALE,
    "temperature_c": TEMP_C,
    "salt_m": SALT_M,
    "dangles": DANGLES,
    "ulength_spec": ULENGTH_SPEC,
    "contribution_spec": CONTRIB_SPEC,
    "accessibility_mode_only": True,
    "use_noLP": USE_noLP,
    "use_b": USE_b,
    "use_interaction_pairwise": USE_interaction_pairwise,
    "use_interaction_first": USE_interaction_first,
    "primary_window_len": PRIMARY_WINDOW_LEN,
    "rank_z_ulengths": AUX_RANK_Z_ULENGTHS,
    "local_z_radius": LOCAL_Z_RADIUS,
    "datasets": [
        {
            "name": d["name"],
            "input_csv": str(d["input_csv"]),
            "timeout_sec_per_seq": d["timeout_sec_per_seq"],
        }
        for d in DATASETS
    ],
}

(OFFICIAL_DIR / "environment.json").write_text(json.dumps(environment_info, ensure_ascii=False, indent=2), encoding="utf-8")
(OFFICIAL_DIR / "run_config.json").write_text(json.dumps(run_config, ensure_ascii=False, indent=2), encoding="utf-8")

print("RNAup version:", RNAUP_VERSION)
print("Python version:", platform.python_version())
print("Official command:", " ".join(OFFICIAL_CMD))
print("Saved:", OFFICIAL_DIR / "environment.json")
print("Saved:", OFFICIAL_DIR / "run_config.json")


RNAup version: RNAup 2.7.2
Python version: 3.12.9
Official command: /opt/anaconda3/envs/rna6nt/bin/RNAup -u 1-20 -c SHIME -T 37.0 --salt 1.021 -d2 --pfScale 1.0 --log-file=RNAup.log --log-time
Saved: /Users/hatanaka/rnaup-official-pf1p00/results/rnaup_official_pf1p00/environment.json
Saved: /Users/hatanaka/rnaup-official-pf1p00/results/rnaup_official_pf1p00/run_config.json


## Runtime snapshot export

This section records the **actual runtime environment** used by this notebook.

It writes a machine-readable snapshot under `artifacts/runtime/`, including:
- Python and platform information
- `RNAup --version`
- `conda env export --no-builds`
- `conda list --explicit`
- current Git revision and Git status (if available)


In [7]:
# Record the actual runtime environment used for this official production run.
# These files are intended to serve as auditable evidence of the environment
# that produced the official RNAup outputs.

RUNTIME_DIR = OFFICIAL_DIR / "artifacts" / "runtime"
RUNTIME_DIR.mkdir(parents=True, exist_ok=True)

def run_cmd(cmd: list[str]) -> dict:
    """Run a command and capture stdout/stderr without raising on failure."""
    try:
        p = subprocess.run(cmd, text=True, capture_output=True, check=False)
        return {
            "command": cmd,
            "returncode": p.returncode,
            "stdout": p.stdout,
            "stderr": p.stderr,
        }
    except Exception as e:
        return {
            "command": cmd,
            "returncode": None,
            "stdout": "",
            "stderr": str(e),
        }

runtime_snapshot = {
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "python_version": platform.python_version(),
    "platform": platform.platform(),
    "machine": platform.machine(),
    "processor": platform.processor(),
    "python_executable": os.sys.executable,
    "cwd": os.getcwd(),
    "project_root": str(PROJECT_ROOT),
    "data_dir": str(DATA_DIR),
    "results_dir": str(RESULTS_DIR),
    "official_dir": str(OFFICIAL_DIR),
    "rna_up_bin": RNAUP_BIN,
}

rna_up_version_info = run_cmd([RNAUP_BIN, "--version"])
conda_env_export_info = run_cmd(["conda", "env", "export", "--name", "rna6nt", "--no-builds"])
conda_explicit_info = run_cmd(["conda", "list", "--explicit", "--name", "rna6nt"])
git_rev_info = run_cmd(["git", "rev-parse", "HEAD"])
git_status_info = run_cmd(["git", "status", "--short"])

(RUNTIME_DIR / "environment_runtime_snapshot.json").write_text(
    json.dumps(runtime_snapshot, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
(RUNTIME_DIR / "RNAup_version.txt").write_text(
    rna_up_version_info["stdout"] or rna_up_version_info["stderr"],
    encoding="utf-8",
)
(RUNTIME_DIR / "conda_env_export.yml").write_text(
    conda_env_export_info["stdout"] or conda_env_export_info["stderr"],
    encoding="utf-8",
)
(RUNTIME_DIR / "conda_explicit.txt").write_text(
    conda_explicit_info["stdout"] or conda_explicit_info["stderr"],
    encoding="utf-8",
)
(RUNTIME_DIR / "git_revision.txt").write_text(
    git_rev_info["stdout"] or git_rev_info["stderr"],
    encoding="utf-8",
)
(RUNTIME_DIR / "git_status.txt").write_text(
    git_status_info["stdout"] or git_status_info["stderr"],
    encoding="utf-8",
)

print(json.dumps(runtime_snapshot, ensure_ascii=False, indent=2))
print("Saved runtime snapshot to:", RUNTIME_DIR)


{
  "timestamp_utc": "2026-04-04T13:01:29.789276+00:00",
  "python_version": "3.12.9",
  "platform": "macOS-26.3.1-arm64-arm-64bit",
  "machine": "arm64",
  "processor": "arm",
  "python_executable": "/opt/anaconda3/envs/rna6nt/bin/python",
  "cwd": "/Users/hatanaka/rnaup-official-pf1p00/notebooks",
  "project_root": "/Users/hatanaka/rnaup-official-pf1p00",
  "data_dir": "/Users/hatanaka/rnaup-official-pf1p00/data",
  "results_dir": "/Users/hatanaka/rnaup-official-pf1p00/results",
  "official_dir": "/Users/hatanaka/rnaup-official-pf1p00/results/rnaup_official_pf1p00",
  "rna_up_bin": "/opt/anaconda3/envs/rna6nt/bin/RNAup"
}
Saved runtime snapshot to: /Users/hatanaka/rnaup-official-pf1p00/results/rnaup_official_pf1p00/artifacts/runtime


## Official RNAup run: CRW

This section generates the official CRW outputs under the fixed final configuration.
The run saves:
- per-sequence raw RNAup outputs,
- per-sequence manifests,
- dataset-level manifest,
- parsed long table,
- parsed wide table,
- dataset status summary.


In [8]:
crw_result = process_dataset(
    dataset_cfg=DATASETS[0],
    official_root=OFFICIAL_DIR,
    rnaup_version=RNAUP_VERSION,
    cmd=OFFICIAL_CMD,
)

print(json.dumps(crw_result["status"], ensure_ascii=False, indent=2))


[CRW] 1/113 : a.23.b.B.japonicum.2
[CRW] 10/113 : b.23.a.T.tenax
[CRW] 20/113 : b.23.b.R.capsulatus
[CRW] 30/113 : b.23.m.L.tarentolae
[CRW] 40/113 : d.23.b.B.burgdorferi
[CRW] 50/113 : d.23.b.E.coli
[CRW] 60/113 : d.23.b.M.genitalium
[CRW] 70/113 : d.23.b.S.ambofaciens
[CRW] 80/113 : d.23.b.S.pyogenes_No.4
[CRW] 90/113 : d.23.e.P.falciparum.S
[CRW] 100/113 : d.23.m.C.lacertina
[CRW] 110/113 : d.23.m.S.sinuspaulianus
[CRW] 113/113 : d.23.y.C.paradoxa
{
  "dataset_name": "CRW",
  "input_csv": "/Users/hatanaka/rnaup-official-pf1p00/data/crw_from_bpseq_acgu.csv",
  "official_command": "/opt/anaconda3/envs/rna6nt/bin/RNAup -u 1-20 -c SHIME -T 37.0 --salt 1.021 -d2 --pfScale 1.0 --log-file=RNAup.log --log-time",
  "RNAup_version": "RNAup 2.7.2",
  "pfScale_used": 1.0,
  "n_sequences_input": 113,
  "n_manifest_rows": 113,
  "n_success": 113,
  "n_failure": 0,
  "n_warning": 0,
  "n_missing_output": 0,
  "n_long_rows": 31508500,
  "n_wide_rows": 314520,
  "raw_root": "/Users/hatanaka/rnaup-of

## Official RNAup run: bpRNA-new

This section generates the official bpRNA-new outputs under the same fixed configuration.
Because the command line is identical to the CRW run, the resulting datasets are directly comparable.


In [9]:
bprna_result = process_dataset(
    dataset_cfg=DATASETS[1],
    official_root=OFFICIAL_DIR,
    rnaup_version=RNAUP_VERSION,
    cmd=OFFICIAL_CMD,
)

print(json.dumps(bprna_result["status"], ensure_ascii=False, indent=2))


[bpRNA-new] 1/5388 : AACY020454584.1_604
[bpRNA-new] 10/5388 : AACY020496190.1_1503
[bpRNA-new] 20/5388 : AAXW01000073.1_3077
[bpRNA-new] 30/5388 : ABPY01006745.1_33
[bpRNA-new] 40/5388 : ACCA01000039.1_31556
[bpRNA-new] 50/5388 : ACII01000060.1_3924
[bpRNA-new] 60/5388 : ACZG01000001.1_1182032
[bpRNA-new] 70/5388 : ADGC01006520.1_5852
[bpRNA-new] 80/5388 : ADLR01000090.1_3100
[bpRNA-new] 90/5388 : AENI01006267.1_25045
[bpRNA-new] 100/5388 : AE004439.1_841385
[bpRNA-new] 110/5388 : AE016830.1_880359
[bpRNA-new] 120/5388 : AE004437.1_1988005
[bpRNA-new] 130/5388 : AE013599.4_1652685
[bpRNA-new] 140/5388 : AE017354.1_3303482
[bpRNA-new] 150/5388 : AFWW01000010.1_2263313
[bpRNA-new] 160/5388 : AGTN01285523.1_140
[bpRNA-new] 170/5388 : AL590842.1_92360
[bpRNA-new] 180/5388 : AL645882.2_1712074
[bpRNA-new] 190/5388 : AM747720.1_2545503
[bpRNA-new] 200/5388 : AP006716.1_2117568
[bpRNA-new] 210/5388 : BAAU01003136.1_1053
[bpRNA-new] 220/5388 : BABB01012728.1_605
[bpRNA-new] 230/5388 : BA00002

## Dataset-level QC

The QC below is intentionally simple and audit-oriented.

It checks whether:
- all input transcripts were processed,
- warnings were absent,
- failures were absent,
- expected output files were created,
- key wide-format features were populated.

This is the minimum QC needed for an official production notebook.


In [10]:
crw_qc_df = make_dataset_qc(crw_result)
bprna_qc_df = make_dataset_qc(bprna_result)
dataset_qc_df = pd.concat([crw_qc_df, bprna_qc_df], ignore_index=True)

dataset_qc_path = OFFICIAL_DIR / "dataset_qc.csv"
dataset_qc_df.to_csv(dataset_qc_path, index=False)

display(dataset_qc_df)
print("Saved:", dataset_qc_path)


,dataset,metric,value
0,CRW,input_transcripts,113.0
1,CRW,manifest_rows,113.0
2,CRW,long_rows,31508500.0
3,CRW,wide_rows,314520.0
4,CRW,warnings,0.0
5,CRW,failures,0.0
6,CRW,missing_output,0.0
7,CRW,missing_rate::rnaup_open4_S,0.0
8,CRW,missing_rate::rnaup_open6_S,0.0
9,CRW,missing_rate::rnaup_open4_rank_in_transcript,0.0


Saved: /Users/hatanaka/rnaup-official-pf1p00/results/rnaup_official_pf1p00/dataset_qc.csv


## Optional lightweight comparison against existing reference runs

If an existing reference long-format run is available under `results/rnaup/<dataset>/...`,
this section computes a lightweight comparison for:
- `rnaup_open4_S`
- `rnaup_open6_S`
- transcript-internal rank
- top-10% overlap

This comparison is optional and does not affect the official output generation.
It is included here only as an additional sanity check.


In [11]:
validation_dir = OFFICIAL_DIR / "validation"
validation_dir.mkdir(parents=True, exist_ok=True)

reference_specs = [
    {
        "dataset_name": "CRW",
        "new_long_df": crw_result["long_df"],
        "reference_long_path": RESULTS_DIR / "rnaup" / "CRW" / "rnaup_parsed_long" / "CRW" / "rnaup_long.csv.gz",
        "prefix": "crw",
    },
    {
        "dataset_name": "bpRNA-new",
        "new_long_df": bprna_result["long_df"],
        "reference_long_path": RESULTS_DIR / "rnaup" / "bpRNA-new" / "rnaup_parsed_long" / "bpRNA-new" / "rnaup_long.csv.gz",
        "prefix": "bprna_new",
    },
]

reference_summary_frames = []

for spec in reference_specs:
    ref_path = spec["reference_long_path"]
    if ref_path.exists():
        detail_df, summary_df = summarize_reference_comparison(
            new_long_df=spec["new_long_df"],
            ref_long_path=ref_path,
            dataset_name=spec["dataset_name"],
        )

        detail_path = validation_dir / f"{spec['prefix']}_vs_reference_detail.csv"
        summary_path = validation_dir / f"{spec['prefix']}_vs_reference_summary.csv"
        detail_df.to_csv(detail_path, index=False)
        summary_df.to_csv(summary_path, index=False)

        reference_summary_frames.append(summary_df)

        print(f"Reference comparison available for {spec['dataset_name']}")
        display(summary_df)
        print("Saved:", detail_path)
        print("Saved:", summary_path)
    else:
        print(f"Reference long file not found for {spec['dataset_name']}: {ref_path}")

if len(reference_summary_frames) > 0:
    reference_summary_df = pd.concat(reference_summary_frames, ignore_index=True)
    reference_summary_all_path = validation_dir / "reference_comparison_summary_all.csv"
    reference_summary_df.to_csv(reference_summary_all_path, index=False)
    print("Saved:", reference_summary_all_path)
else:
    reference_summary_df = pd.DataFrame()


Reference long file not found for CRW: /Users/hatanaka/rnaup-official-pf1p00/results/rnaup/CRW/rnaup_parsed_long/CRW/rnaup_long.csv.gz
Reference long file not found for bpRNA-new: /Users/hatanaka/rnaup-official-pf1p00/results/rnaup/bpRNA-new/rnaup_parsed_long/bpRNA-new/rnaup_long.csv.gz


## Final acceptance summary

This section determines whether the newly generated datasets can be accepted as the official RNAup outputs.

Acceptance criteria used here:
- warning count = 0
- failure count = 0
- missing output count = 0
- input transcript count matches manifest row count


In [12]:
def is_clean_official_run(result: dict) -> bool:
    status = result["status"]
    return (
        status["n_warning"] == 0 and
        status["n_failure"] == 0 and
        status["n_missing_output"] == 0 and
        status["n_sequences_input"] == status["n_manifest_rows"]
    )

crw_clean = is_clean_official_run(crw_result)
bprna_clean = is_clean_official_run(bprna_result)
official_accept = crw_clean and bprna_clean

final_lines = []
final_lines.append("Official RNAup final assessment")
final_lines.append("=" * 32)
final_lines.append("")
final_lines.append(f"Official output root: {OFFICIAL_DIR}")
final_lines.append(f"RNAup version: {RNAUP_VERSION}")
final_lines.append(f"Official command: {' '.join(OFFICIAL_CMD)}")
final_lines.append("")
final_lines.append("Dataset-level acceptance:")
final_lines.append(f"- CRW clean run: {crw_clean}")
final_lines.append(f"- bpRNA-new clean run: {bprna_clean}")
final_lines.append("")
final_lines.append("CRW status summary:")
for k, v in crw_result["status"].items():
    final_lines.append(f"  - {k}: {v}")
final_lines.append("")
final_lines.append("bpRNA-new status summary:")
for k, v in bprna_result["status"].items():
    final_lines.append(f"  - {k}: {v}")
final_lines.append("")
final_lines.append(f"Official RNAup adoption under pfScale=1.00: {official_accept}")

final_txt = "\n".join(final_lines)

final_assessment_path = OFFICIAL_DIR / "final_assessment.txt"
final_assessment_json_path = OFFICIAL_DIR / "final_assessment.json"
run_summary_csv_path = OFFICIAL_DIR / "run_summary.csv"
run_summary_txt_path = OFFICIAL_DIR / "run_summary.txt"

run_summary_df = pd.DataFrame([crw_result["status"], bprna_result["status"]])
run_summary_df.to_csv(run_summary_csv_path, index=False)
run_summary_txt_path.write_text(run_summary_df.to_string(index=False), encoding="utf-8")

final_assessment_path.write_text(final_txt, encoding="utf-8")
final_assessment_json_path.write_text(
    json.dumps(
        {
            "official_output_root": str(OFFICIAL_DIR),
            "RNAup_version": RNAUP_VERSION,
            "official_command": " ".join(OFFICIAL_CMD),
            "crw_clean": crw_clean,
            "bprna_clean": bprna_clean,
            "official_accept": official_accept,
            "crw_status": crw_result["status"],
            "bprna_status": bprna_result["status"],
        },
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

print(final_txt)
print("Saved:", run_summary_csv_path)
print("Saved:", run_summary_txt_path)
print("Saved:", final_assessment_path)
print("Saved:", final_assessment_json_path)


Official RNAup final assessment

Official output root: /Users/hatanaka/rnaup-official-pf1p00/results/rnaup_official_pf1p00
RNAup version: RNAup 2.7.2
Official command: /opt/anaconda3/envs/rna6nt/bin/RNAup -u 1-20 -c SHIME -T 37.0 --salt 1.021 -d2 --pfScale 1.0 --log-file=RNAup.log --log-time

Dataset-level acceptance:
- CRW clean run: True
- bpRNA-new clean run: True

CRW status summary:
  - dataset_name: CRW
  - input_csv: /Users/hatanaka/rnaup-official-pf1p00/data/crw_from_bpseq_acgu.csv
  - official_command: /opt/anaconda3/envs/rna6nt/bin/RNAup -u 1-20 -c SHIME -T 37.0 --salt 1.021 -d2 --pfScale 1.0 --log-file=RNAup.log --log-time
  - RNAup_version: RNAup 2.7.2
  - pfScale_used: 1.0
  - n_sequences_input: 113
  - n_manifest_rows: 113
  - n_success: 113
  - n_failure: 0
  - n_warning: 0
  - n_missing_output: 0
  - n_long_rows: 31508500
  - n_wide_rows: 314520
  - raw_root: /Users/hatanaka/rnaup-official-pf1p00/results/rnaup_official_pf1p00/CRW/rnaup_raw/CRW
  - manifest_path: /Users/

## Output inventory

The final cell prints the key files produced by the official notebook so that a third party can quickly find:
- configuration files,
- per-dataset manifest files,
- long tables,
- wide tables,
- final acceptance summary.


In [13]:
key_files = [
    OFFICIAL_DIR / "environment.json",
    OFFICIAL_DIR / "run_config.json",
    OFFICIAL_DIR / "dataset_qc.csv",
    OFFICIAL_DIR / "run_summary.csv",
    OFFICIAL_DIR / "run_summary.txt",
    OFFICIAL_DIR / "final_assessment.txt",
    OFFICIAL_DIR / "final_assessment.json",

    OFFICIAL_DIR / "CRW" / "meta" / "input_transcript_metadata.csv",
    OFFICIAL_DIR / "CRW" / "meta" / "rnaup_run_manifest.csv",
    OFFICIAL_DIR / "CRW" / "rnaup_parsed_long" / "CRW" / "rnaup_long.csv.gz",
    OFFICIAL_DIR / "CRW" / "rnaup_parsed_wide" / "CRW" / "rnaup_window_features.csv.gz",

    OFFICIAL_DIR / "bpRNA-new" / "meta" / "input_transcript_metadata.csv",
    OFFICIAL_DIR / "bpRNA-new" / "meta" / "rnaup_run_manifest.csv",
    OFFICIAL_DIR / "bpRNA-new" / "rnaup_parsed_long" / "bpRNA-new" / "rnaup_long.csv.gz",
    OFFICIAL_DIR / "bpRNA-new" / "rnaup_parsed_wide" / "bpRNA-new" / "rnaup_window_features.csv.gz",
]

inventory_rows = []
for p in key_files:
    inventory_rows.append({
        "path": str(p),
        "exists": p.exists(),
        "size_bytes": None if not p.exists() else p.stat().st_size,
    })

inventory_df = pd.DataFrame(inventory_rows)
display(inventory_df)


,path,exists,size_bytes
0,/Users/hatanaka/rnaup-official-pf1p00/results/...,True,441
1,/Users/hatanaka/rnaup-official-pf1p00/results/...,True,962
2,/Users/hatanaka/rnaup-official-pf1p00/results/...,True,1061
3,/Users/hatanaka/rnaup-official-pf1p00/results/...,True,1600
4,/Users/hatanaka/rnaup-official-pf1p00/results/...,True,2426
5,/Users/hatanaka/rnaup-official-pf1p00/results/...,True,2485
6,/Users/hatanaka/rnaup-official-pf1p00/results/...,True,2519
7,/Users/hatanaka/rnaup-official-pf1p00/results/...,True,643469
8,/Users/hatanaka/rnaup-official-pf1p00/results/...,True,172862
9,/Users/hatanaka/rnaup-official-pf1p00/results/...,True,755265544
